### XAI com Deep Learning
Para comparar com o 05_XAI

In [1]:
!pip install -q gcsfs duckdb
import pandas as pd
import gcsfs
from google.colab import auth
import duckdb

# 1. Garante a autenticação nativa do Colab
auth.authenticate_user()

# 2. Inicializa o FileSystem do GCS apontando para o seu projeto
project_id = 'doutorado-501917'
bucket_name = '2025_rides'
fs = gcsfs.GCSFileSystem(project=project_id)

# 3. Integração Mágica: Registra o gcsfs no DuckDB
# Isso faz o DuckDB usar a autenticação do Colab automaticamente
duckdb.register_filesystem(fs)

# 4. Usa o glob do gcsfs para encontrar todos os arquivos parquet na pasta
file_pattern = f"gs://{bucket_name}/outputs_simulation_V6_3/trips_log/_staging/**/*.parquet"
print(f"Buscando arquivos com o padrão: {file_pattern}...")
file_list = fs.glob(file_pattern)
print(f"Encontrados {len(file_list)} arquivos Parquet. Preparando leitura...")

# Adiciona o prefixo gs:// para o DuckDB reconhecer corretamente usando o fsspec/gcsfs
gs_file_list = [f"gs://{f}" for f in file_list]



Buscando arquivos com o padrão: gs://2025_rides/outputs_simulation_V6_3/trips_log/_staging/**/*.parquet...
Encontrados 3531 arquivos Parquet. Preparando leitura...


In [2]:

# 5. Habilita a barra de progresso do DuckDB
duckdb.sql("PRAGMA enable_progress_bar;")
duckdb.sql("PRAGMA enable_print_progress_bar;")

print("Carregando os dados com DuckDB...")

# 6. Cria a query passando a lista de arquivos e executa retornando para DataFrame pandas
# Limitando a saída de string para evitar erros de formatação na query
files_sql_array = ", ".join([f"'{f}'" for f in gs_file_list])

query = f"""
    SELECT *
    FROM read_parquet([{files_sql_array}])
"""

#df = duckdb.sql(query).df()

#print(f"\nSuccessfully read {len(df):,} rows from GCS parquet files using DuckDB.")
#display(df.head())

Carregando os dados com DuckDB...


# INÍCIO DA LEITURA, TRATAMENTO E ANÁLISE DOS DADOS

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import shutil
datasets_dir = "/content/drive/MyDrive/DOUTORADO/DATASETS/DATASETS_PRONTOS"
shutil.copy("/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/arquivos_base/dados_meteorologicos_utci_horario.csv", "./")
shutil.copy("/content/drive/MyDrive/DOUTORADO/003_DADOS_SINTETICOS/arquivos_base/DADOS_AEROPORTO/03_voos_atrasados_sbpa.csv", "./")
shutil.copy(f"{datasets_dir}/Aeroporto_Salgado_Filho_h3_res12.csv", "./")

import pandas as pd

# Carregando os datasets
df_clima = pd.read_csv('/content/dados_meteorologicos_utci_horario.csv')
df_voos = pd.read_csv('/content/03_voos_atrasados_sbpa.csv', sep=";")
df_h3 = pd.read_csv('/content/Aeroporto_Salgado_Filho_h3_res12.csv')

# Exibindo as primeiras linhas de cada um para verificar
print("--- Dados Meteorológicos ---")
display(df_clima.head())

print("\n--- Voos Atrasados ---")
display(df_voos.head())

print("\n--- Aeroporto Salgado Filho (H3) ---")
display(df_h3.head())

--- Dados Meteorológicos ---


,time,temperature_2m,relative_humidity_2m,wind_speed_10m,shortwave_radiation,direct_radiation,diffuse_radiation,direct_normal_irradiance,sunshine_duration,cloud_cover,...,utci_tr_source,utci_wind_speed_10m_mps_used,utci_wind_was_clipped,utci_c,utci_stress_category,utci_discomfort_score_0_100,utci_has_heat_stress,utci_has_cold_stress,utci_has_strong_heat_stress,utci_has_strong_cold_stress
0,2025-01-01 00:00:00,21.4,95,3.46,0.0,0.0,0.0,0.0,0.0,2,...,estimated_from_open_meteo_solar_radiation,3.46,False,19.796398,no thermal stress,0.0,False,False,False,False
1,2025-01-01 01:00:00,21.2,95,3.13,0.0,0.0,0.0,0.0,0.0,1,...,estimated_from_open_meteo_solar_radiation,3.13,False,20.051337,no thermal stress,0.0,False,False,False,False
2,2025-01-01 02:00:00,21.1,95,2.40,0.0,0.0,0.0,0.0,0.0,1,...,estimated_from_open_meteo_solar_radiation,2.40,False,21.135266,no thermal stress,0.0,False,False,False,False
3,2025-01-01 03:00:00,21.0,96,2.30,0.0,0.0,0.0,0.0,0.0,1,...,estimated_from_open_meteo_solar_radiation,2.30,False,21.236648,no thermal stress,0.0,False,False,False,False
4,2025-01-01 04:00:00,20.8,97,2.01,0.0,0.0,0.0,0.0,0.0,4,...,estimated_from_open_meteo_solar_radiation,2.01,False,21.518915,no thermal stress,0.0,False,False,False,False



--- Voos Atrasados ---


,ICAO_EMPRESA_AEREA,NUMERO_VOO,CODIGO_AUTORIZACAO_DI,CODIGO_TIPO_LINHA,ICAO_AERODROMO_ORIGEM,ICAO_AERODROMO_DESTINO,PARTIDA_PREVISTA,PARTIDA_REAL,CHEGADA_PREVISTA,CHEGADA_REAL,SITUACAO_VOO,CODIGO_JUSTIFICATIVA,ATRASO_MINUTOS,STATUS_ATRASO,DATA_PREVISTA,DIA_SEMANA,NOME_DIA,HORA_PREVISTA
0,GLO,1885,0,N,SBGR,SBPA,2025-01-23 14:50:00,2025-01-23 15:09:00,2025-01-23 16:35:00,2025-01-23 16:57:00,REALIZADO,NaN,22.0,Atraso Leve (15-59 min),2025-01-23,3,Quinta,16
1,GLO,1885,0,N,SBGR,SBPA,2025-01-26 14:50:00,2025-01-26 15:12:00,2025-01-26 16:35:00,2025-01-26 16:59:00,REALIZADO,NaN,24.0,Atraso Leve (15-59 min),2025-01-26,6,Domingo,16
2,GLO,1885,0,N,SBGR,SBPA,2025-01-29 14:50:00,2025-01-29 14:50:00,2025-01-29 16:35:00,2025-01-29 17:07:00,REALIZADO,NaN,32.0,Atraso Leve (15-59 min),2025-01-29,2,Quarta,16
3,LPE,2422,0,I,SPJC,SBPA,2025-01-06 01:50:00,2025-01-06 02:38:00,2025-01-06 06:30:00,2025-01-06 07:05:00,REALIZADO,NaN,35.0,Atraso Leve (15-59 min),2025-01-06,0,Segunda,6
4,LPE,2422,0,I,SPJC,SBPA,2025-01-20 01:50:00,2025-01-20 02:33:00,2025-01-20 06:30:00,2025-01-20 07:01:00,REALIZADO,NaN,31.0,Atraso Leve (15-59 min),2025-01-20,0,Segunda,6



--- Aeroporto Salgado Filho (H3) ---


,h3_index
0,8ca90e935c895ff
1,8ca90e935d2ebff
2,8ca90129b6d31ff
3,8ca90e9264e69ff
4,8ca90129b30b5ff


In [6]:
import gcsfs
import duckdb

# Garante que o filesystem gcsfs esteja registrado no DuckDB
fs = gcsfs.GCSFileSystem(project=project_id)
try:
    duckdb.register_filesystem(fs)
except Exception:
    pass # Ignora caso já esteja registrado

# Caminhos base das simulações
caminhos_base = [
    f'gs://{bucket_name}/outputs_simulation_V6_3/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_4/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_5/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_6/trips_log/_staging/**/*.parquet',
    f'gs://{bucket_name}/outputs_simulation_V6_7/trips_log/_staging/**/*.parquet',
]

print("Buscando todos os arquivos Parquet nos diretórios (isso pode levar alguns instantes)...")
todos_arquivos = []
for caminho in caminhos_base:
    arquivos = fs.glob(caminho)
    todos_arquivos.extend([f"gs://{f}" for f in arquivos])

print(f"Total de {len(todos_arquivos):,} arquivos Parquet encontrados.")

# Criando o array de strings para injetar na query do DuckDB
files_sql_array = ", ".join([f"'{f}'" for f in todos_arquivos])


Buscando todos os arquivos Parquet nos diretórios (isso pode levar alguns instantes)...
Total de 12,368 arquivos Parquet encontrados.


In [11]:
print("Preparando a query principal para executar com DuckDB...")
print("Lendo do GCS com a nova integração e aplicando undersampling...")

# Habilita a barra de progresso do DuckDB
duckdb.sql("PRAGMA enable_progress_bar;")
duckdb.sql("PRAGMA enable_print_progress_bar;")

# A query balanceada
inicio_ts = 1735699200
fim_ts = 1767235200

# Combine all conditions into a single query with appropriate sampling
# Substituímos {caminhos_parquet} por [{files_sql_array}]
query_combined = f"""
    SELECT
        request_ts,
        event_name,
        origin_h3,
        CASE
            WHEN UPPER(event_name) LIKE '%ATRASADO%' THEN 'DS_VOO'
            WHEN UPPER(event_name) LIKE '%SEVERIDADE%' THEN 'DS_CLIMA'
            WHEN  event_name IS NULL THEN 'NULO'
            ELSE 'DS_OUTROS'
        END AS dataset_type
    FROM read_parquet([{files_sql_array}], hive_partitioning = true)
    WHERE request_ts >= {inicio_ts}
      AND request_ts < {fim_ts}
    USING SAMPLE 30 PERCENT
"""


Preparando a query principal para executar com DuckDB...
Lendo do GCS com a nova integração e aplicando undersampling...


In [12]:
# Instala a biblioteca necessária
!pip install google-cloud-storage

from google.cloud import storage
from google.colab import auth

# Autenticação
auth.authenticate_user()

# Crie um cliente apontando para o seu projeto
project_id = 'doutorado-501917'
client = storage.Client(project=project_id)

# Acesse o bucket e o arquivo específico
bucket_name = '2025_rides'
bucket = client.get_bucket(bucket_name)

In [13]:
# 5. Habilita a barra de progresso do DuckDB
duckdb.sql("PRAGMA enable_progress_bar;")
duckdb.sql("PRAGMA enable_print_progress_bar;")

print("Carregando os dados com DuckDB...")

# 6. Cria a query passando a lista de arquivos e executa retornando para DataFrame pandas
# Limitando a saída de string para evitar erros de formatação na query
files_sql_array = ", ".join([f"'{f}'" for f in gs_file_list])

df = duckdb.sql(query_combined).df()

print(f"\nSuccessfully read {len(df):,} rows from GCS parquet files using DuckDB.")
display(df.head())

df_combined = duckdb.sql(query_combined).df()

# Filtra para criar os datasets DS_VOO, DS_CLIMA e DS_OUTROS
DS_VOO = df_combined[df_combined['dataset_type'] == 'DS_VOO'].drop(columns=['dataset_type'])
DS_CLIMA = df_combined[df_combined['dataset_type'] == 'DS_CLIMA'].drop(columns=['dataset_type'])
DS_NULO = df_combined[df_combined['dataset_type'] == 'NULO'].drop(columns=['dataset_type'])
DS_OUTROS = df_combined[df_combined['dataset_type'] == 'DS_OUTROS'].drop(columns=['dataset_type'])

print(f"🚀 DATASET DE ATRASOS (DS_VOO): {len(DS_VOO):,}")
print(f"🚀 DATASET DE ATRASOS (DS_CLIMA): {len(DS_CLIMA):,}")
print(f"🚀 DATASET DE NULO: {len(DS_NULO):,}")
print(f"🚀 DATASET DE ATRASOS (DS_OUTROS): {len(DS_OUTROS):,}")
display(DS_VOO.head())

Carregando os dados com DuckDB...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Successfully read 17,554,961 rows from GCS parquet files using DuckDB.


,request_ts,event_name,origin_h3,dataset_type
0,1735740599,,8ca901645a653ff,DS_OUTROS
1,1735742831,,8ca9010cb39d3ff,DS_OUTROS
2,1735744825,,8ca9016443407ff,DS_OUTROS
3,1735746431,,8ca901398704dff,DS_OUTROS
4,1735747630,,8ca90129279d7ff,DS_OUTROS


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

🚀 DATASET DE ATRASOS (DS_VOO): 25,342
🚀 DATASET DE ATRASOS (DS_CLIMA): 135,770
🚀 DATASET DE NULO: 0
🚀 DATASET DE ATRASOS (DS_OUTROS): 16,879,980


,request_ts,event_name,origin_h3
6608,1735929003,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9358987ff
7413,1735930958,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e935cc6dff
7583,1735898901,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9265543ff
8115,1735900335,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e934ab65ff
8521,1735898736,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e934b5c1ff


In [14]:
import pandas as pd
import numpy as np

print("Iniciando a normalização do tamanho dos datasets DS_VOO, DS_CLIMA e DS_OUTROS...")

# Definir o período de interesse para a distribuição (2025 inteiro)
start_date = pd.to_datetime('2025-01-01 00:00:00')
end_date = pd.to_datetime('2025-12-31 23:59:59')

# Convert UNIX timestamps in seconds to datetime objects
DS_VOO['request_ts_dt'] = pd.to_datetime(DS_VOO['request_ts'], unit='s')
DS_CLIMA['request_ts_dt'] = pd.to_datetime(DS_CLIMA['request_ts'], unit='s')
DS_OUTROS['request_ts_dt'] = pd.to_datetime(DS_OUTROS['request_ts'], unit='s')

print("Corrected datetime format:")
display(DS_VOO[['request_ts', 'request_ts_dt']].head())

Iniciando a normalização do tamanho dos datasets DS_VOO, DS_CLIMA e DS_OUTROS...
Corrected datetime format:


,request_ts,request_ts_dt
6608,1735929003,2025-01-03 18:30:03
7413,1735930958,2025-01-03 19:02:38
7583,1735898901,2025-01-03 10:08:21
8115,1735900335,2025-01-03 10:32:15
8521,1735898736,2025-01-03 10:05:36


In [15]:
# Filtrar cada DataFrame para o período de 2025
print(f"Filtrando datasets para o período de {start_date.strftime('%Y-%m-%d')} a {end_date.strftime('%Y-%m-%d')}...")
DS_VOO_2025 = DS_VOO[(DS_VOO['request_ts_dt'] >= start_date) & (DS_VOO['request_ts_dt'] <= end_date)]
DS_CLIMA_2025 = DS_CLIMA[(DS_CLIMA['request_ts_dt'] >= start_date) & (DS_CLIMA['request_ts_dt'] <= end_date)]
DS_OUTROS_2025 = DS_OUTROS[(DS_OUTROS['request_ts_dt'] >= start_date) & (DS_OUTROS['request_ts_dt'] <= end_date)]

print(f"Tamanho de DS_VOO_2025 após filtro: {len(DS_VOO_2025):,} linhas")
print(f"Tamanho de DS_CLIMA_2025 após filtro: {len(DS_CLIMA_2025):,} linhas")
print(f"Tamanho de DS_OUTROS_2025 após filtro: {len(DS_OUTROS_2025):,} linhas")

# Encontrar o menor tamanho entre os DataFrames filtrados
min_size = min(len(DS_VOO_2025), len(DS_CLIMA_2025), len(DS_OUTROS_2025))
print(f"\nO menor tamanho entre os datasets filtrados é: {min_size:,} linhas.")

# Realizar o subsampling (amostragem aleatória) para equalizar o tamanho, mantendo a distribuição temporal
print(f"Subsampling todos os datasets para {min_size:,} linhas...")

if len(DS_VOO_2025) > min_size:
    DS_VOO_NORMALIZED = DS_VOO_2025.sample(n=min_size, random_state=42).sort_values('request_ts_dt').reset_index(drop=True)
else:
    DS_VOO_NORMALIZED = DS_VOO_2025.sort_values('request_ts_dt').reset_index(drop=True)

if len(DS_CLIMA_2025) > min_size:
    DS_CLIMA_NORMALIZED = DS_CLIMA_2025.sample(n=min_size, random_state=42).sort_values('request_ts_dt').reset_index(drop=True)
else:
    DS_CLIMA_NORMALIZED = DS_CLIMA_2025.sort_values('request_ts_dt').reset_index(drop=True)

if len(DS_OUTROS_2025) > min_size:
    DS_OUTROS_NORMALIZED = DS_OUTROS_2025.sample(n=min_size, random_state=42).sort_values('request_ts_dt').reset_index(drop=True)
else:
    DS_OUTROS_NORMALIZED = DS_OUTROS_2025.sort_values('request_ts_dt').reset_index(drop=True)

# Atualizar os DataFrames originais com os resultados normalizados
DS_VOO = DS_VOO_NORMALIZED
DS_CLIMA = DS_CLIMA_NORMALIZED
DS_OUTROS = DS_OUTROS_NORMALIZED

print("\n✅ Normalização concluída!")
print(f"Novo tamanho de DS_VOO: {len(DS_VOO):,} linhas")
print(f"Novo tamanho de DS_CLIMA: {len(DS_CLIMA):,} linhas")
print(f"Novo tamanho de DS_OUTROS: {len(DS_OUTROS):,} linhas")

print("\nVerificação das primeiras linhas de cada dataset normalizado:")
print("\nDS_VOO (Normalizado):")
display(DS_VOO.head())

print("\nDS_CLIMA (Normalizado):")
display(DS_CLIMA.head())

print("\nDS_OUTROS (Normalizado):")
display(DS_OUTROS.head())

Filtrando datasets para o período de 2025-01-01 a 2025-12-31...
Tamanho de DS_VOO_2025 após filtro: 25,342 linhas
Tamanho de DS_CLIMA_2025 após filtro: 135,770 linhas
Tamanho de DS_OUTROS_2025 após filtro: 16,879,980 linhas

O menor tamanho entre os datasets filtrados é: 25,342 linhas.
Subsampling todos os datasets para 25,342 linhas...

✅ Normalização concluída!
Novo tamanho de DS_VOO: 25,342 linhas
Novo tamanho de DS_CLIMA: 25,342 linhas
Novo tamanho de DS_OUTROS: 25,342 linhas

Verificação das primeiras linhas de cada dataset normalizado:

DS_VOO (Normalizado):


,request_ts,event_name,origin_h3,request_ts_dt
0,1735898418,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9264007ff,2025-01-03 10:00:18
1,1735898430,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9341731ff,2025-01-03 10:00:30
2,1735898468,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9264e67ff,2025-01-03 10:01:08
3,1735898477,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9358c3dff,2025-01-03 10:01:17
4,1735898493,Aeroporto Salgado Filho Voo Atrasado- SBPA,8ca90e9342583ff,2025-01-03 10:01:33



DS_CLIMA (Normalizado):


,request_ts,event_name,origin_h3,request_ts_dt
0,1735718419,Aeroporto Salgado Filho Weather Severidade 5,8ca90e93409c5ff,2025-01-01 08:00:19
1,1735718429,Aeroporto Salgado Filho Weather Severidade 5,8ca90e926458dff,2025-01-01 08:00:29
2,1735718447,Aeroporto Salgado Filho Weather Severidade 5,8ca90e935ce21ff,2025-01-01 08:00:47
3,1735718525,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9341801ff,2025-01-01 08:02:05
4,1735718754,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9264081ff,2025-01-01 08:05:54



DS_OUTROS (Normalizado):


,request_ts,event_name,origin_h3,request_ts_dt
0,1735704463,,8ca901772a715ff,2025-01-01 04:07:43
1,1735706555,,8ca9016606b33ff,2025-01-01 04:42:35
2,1735709780,,8ca9012aecc43ff,2025-01-01 05:36:20
3,1735710905,,8ca90139a31c7ff,2025-01-01 05:55:05
4,1735711243,,8ca90e91ccac9ff,2025-01-01 06:00:43


In [16]:
print(f"🚀 DATASET DE EVENTOS CLIMÁTICOS (DS_CLIMA): {len(DS_CLIMA):,}")
display(DS_CLIMA.head())

🚀 DATASET DE EVENTOS CLIMÁTICOS (DS_CLIMA): 25,342


,request_ts,event_name,origin_h3,request_ts_dt
0,1735718419,Aeroporto Salgado Filho Weather Severidade 5,8ca90e93409c5ff,2025-01-01 08:00:19
1,1735718429,Aeroporto Salgado Filho Weather Severidade 5,8ca90e926458dff,2025-01-01 08:00:29
2,1735718447,Aeroporto Salgado Filho Weather Severidade 5,8ca90e935ce21ff,2025-01-01 08:00:47
3,1735718525,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9341801ff,2025-01-01 08:02:05
4,1735718754,Aeroporto Salgado Filho Weather Severidade 5,8ca90e9264081ff,2025-01-01 08:05:54


In [17]:
# Executa a query e converte diretamente para DataFrame Pandas
print(f"🚀 DATASET DE OUTROS EVENTOS (DS_OUTROS): {len(DS_OUTROS):,}")
display(DS_OUTROS.head())

🚀 DATASET DE OUTROS EVENTOS (DS_OUTROS): 25,342


,request_ts,event_name,origin_h3,request_ts_dt
0,1735704463,,8ca901772a715ff,2025-01-01 04:07:43
1,1735706555,,8ca9016606b33ff,2025-01-01 04:42:35
2,1735709780,,8ca9012aecc43ff,2025-01-01 05:36:20
3,1735710905,,8ca90139a31c7ff,2025-01-01 05:55:05
4,1735711243,,8ca90e91ccac9ff,2025-01-01 06:00:43


In [20]:
len_atraso = len(DS_VOO)
len_climatico = len(DS_CLIMA)
len_nao_evento = len(DS_OUTROS)

total = len_atraso + len_climatico + len_nao_evento

print(f"Total de linhas: {total:,}\n")
print(f"Atraso de Voo (DS_VOO): {len_atraso:,} ({len_atraso/total:.2%})")
print(f"Evento Climático (DS_CLIMA): {len_climatico:,} ({len_climatico/total:.2%})")
print(f"Outros Eventos (DS_OUTROS): {len_nao_evento:,} ({len_nao_evento/total:.2%})")

Total de linhas: 76,026

Atraso de Voo (DS_VOO): 25,342 (33.33%)
Evento Climático (DS_CLIMA): 25,342 (33.33%)
Outros Eventos (DS_OUTROS): 25,342 (33.33%)


## 🔧 Feature Engineering Avançado

Reconstrução das features a partir dos datasets balanceados (`DS_VOO`, `DS_CLIMA`, `DS_OUTROS`).
Adicionamos sinais que o LightGBM do `05_XAI` **não utilizou**:

- **Espaciais:** decodificação do H3 → latitude/longitude + distância ao Aeroporto Salgado Filho
- **Cíclicas:** hora, mês e dia da semana via seno/cosseno (respeitam a natureza circular do tempo)
- **Lags e tendências:** clima das 4h anteriores e sua variação

A hipótese: eventos de voo/clima concentram-se no aeroporto, enquanto "outros" se espalham pela cidade — a localização é o sinal mais discriminativo.

In [ ]:
import pandas as pd
import numpy as np

print("1. Padronizando colunas temporais e janelas de 4h...")
df_clima['time'] = pd.to_datetime(df_clima['time'])
df_voos['CHEGADA_REAL'] = pd.to_datetime(df_voos['CHEGADA_REAL'], errors='coerce')
df_clima['time_window'] = df_clima['time'].dt.floor('4h')
df_voos['time_window']  = df_voos['CHEGADA_REAL'].dt.floor('4h')

for _df in [DS_OUTROS, DS_CLIMA, DS_VOO]:
    _df['request_ts_dt'] = pd.to_datetime(_df['request_ts_dt'])
    _df['time_window']   = _df['request_ts_dt'].dt.floor('4h')

print("2. Definindo TARGET (0=Nao Evento, 1=Clima, 2=Voo)...")
DS_OUTROS['target'] = 0
DS_CLIMA['target']  = 1
DS_VOO['target']    = 2
target_map = {0: '0 (Nao Evento)', 1: '1 (Evento Climatico)', 2: '2 (Atraso de Voo)'}

df_eventos = pd.concat([
    DS_OUTROS[['time_window', 'origin_h3', 'target', 'request_ts_dt']],
    DS_CLIMA [['time_window', 'origin_h3', 'target', 'request_ts_dt']],
    DS_VOO   [['time_window', 'origin_h3', 'target', 'request_ts_dt']],
], ignore_index=True)

print("3. Agregando clima e voos por janela de 4h...")
df_clima_agg = df_clima.drop(columns=['time']).groupby('time_window').mean(numeric_only=True).reset_index()
df_voos_agg = df_voos.dropna(subset=['time_window']).groupby('time_window').agg(
    qtd_voos_previstos  = ('NUMERO_VOO',         'count'),
    qtd_empresas_aereas = ('ICAO_EMPRESA_AEREA', lambda x: x.nunique()),
).reset_index()

print("4. Montando a Base Master...")
df_master = pd.merge(df_eventos, df_clima_agg, on='time_window', how='left')
df_master = pd.merge(df_master, df_voos_agg, on='time_window', how='left')
df_master['qtd_voos_previstos']  = df_master['qtd_voos_previstos'].fillna(0)
df_master['qtd_empresas_aereas'] = df_master['qtd_empresas_aereas'].fillna(0)
df_master = df_master.sort_values('time_window').reset_index(drop=True)

print(f"\n✅ Base Master: {len(df_master):,} linhas x {df_master.shape[1]} colunas")
print(df_master['target'].map(target_map).value_counts())
display(df_master.head())

In [ ]:
!pip install -q h3
import h3
import numpy as np

print("Decodificando H3 -> lat/lng e calculando distancia ao aeroporto...")

# Aeroporto Salgado Filho (SBPA)
AER_LAT, AER_LNG = -29.9939, -51.1711

def h3_to_latlng(h):
    try:
        return h3.cell_to_latlng(h)   # h3 v4
    except AttributeError:
        return h3.h3_to_geo(h)        # h3 v3

def haversine_km(lat1, lng1, lat2, lng2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi   = np.radians(lat2 - lat1)
    dlmb   = np.radians(lng2 - lng1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlmb/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

# Mapeia apenas H3 unicos (eficiencia)
coords = {}
for h in df_master['origin_h3'].dropna().unique():
    try:
        coords[h] = h3_to_latlng(h)
    except Exception:
        coords[h] = (np.nan, np.nan)

df_master['lat'] = df_master['origin_h3'].map(lambda h: coords.get(h, (np.nan, np.nan))[0])
df_master['lng'] = df_master['origin_h3'].map(lambda h: coords.get(h, (np.nan, np.nan))[1])
df_master['dist_aeroporto_km'] = haversine_km(df_master['lat'], df_master['lng'], AER_LAT, AER_LNG)

print("\nDistancia media ao aeroporto por classe (km):")
print(df_master.groupby('target')['dist_aeroporto_km'].mean().rename(index=target_map))
display(df_master[['origin_h3', 'lat', 'lng', 'dist_aeroporto_km', 'target']].head())

In [ ]:
import numpy as np

print("1. Codificacao ciclica temporal (sin/cos)...")
df_master['hora']       = df_master['time_window'].dt.hour
df_master['mes']        = df_master['time_window'].dt.month
df_master['dia_semana'] = df_master['time_window'].dt.dayofweek

df_master['hora_sin'] = np.sin(2*np.pi * df_master['hora'] / 24)
df_master['hora_cos'] = np.cos(2*np.pi * df_master['hora'] / 24)
df_master['mes_sin']  = np.sin(2*np.pi * df_master['mes']  / 12)
df_master['mes_cos']  = np.cos(2*np.pi * df_master['mes']  / 12)
df_master['dia_sin']  = np.sin(2*np.pi * df_master['dia_semana'] / 7)
df_master['dia_cos']  = np.cos(2*np.pi * df_master['dia_semana'] / 7)

print("2. Lags climaticos (4h antes)...")
df_clima_agg = df_clima_agg.sort_values('time_window')
cols_lag = ['temperature_2m', 'relative_humidity_2m', 'wind_speed_10m']
if 'surface_pressure' in df_clima_agg.columns:
    cols_lag.append('surface_pressure')
for col in cols_lag:
    df_clima_agg[f'{col}_lag4h'] = df_clima_agg[col].shift(1)
cols_merge = ['time_window'] + [f'{c}_lag4h' for c in cols_lag]
df_master = pd.merge(df_master, df_clima_agg[cols_merge], on='time_window', how='left')

print("3. Tendencias climaticas (variacao = atual - 4h antes)...")
for col in cols_lag:
    if col in df_master.columns and f'{col}_lag4h' in df_master.columns:
        df_master[f'diff_{col}'] = df_master[col] - df_master[f'{col}_lag4h']

print("\n✅ Features temporais, de lag e de tendencia criadas.")
display(df_master[['time_window', 'hora_sin', 'hora_cos', 'mes_sin', 'dia_sin']].head())

## 🧠 Deep Learning Otimizado

Melhorias sobre o baseline que colapsou (recall da classe 0 = 0.13):

| Aspecto | Baseline | Otimizado |
|---|---|---|
| Split | aleatório (risco de vazamento entre janelas) | **temporal** (treina no passado, testa no futuro) |
| Features | 54 (sem H3) | + **espaciais H3** + **cíclicas** + tendências |
| Arquitetura | Dense+ReLU+BN | **Dense→BN→ReLU→Dropout** (ordem canônica), mais larga |
| Loss | crossentropy | crossentropy + **label smoothing** |
| Otimizador | Adam | **AdamW** (weight decay) |
| LR | fixo | **ReduceLROnPlateau** |

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import StandardScaler

print("--- Preparacao dos dados ---")

# Colunas que NAO sao features (identificadores, alvo, temporais crus)
excluir = ['time_window', 'request_ts_dt', 'event_name', 'origin_h3', 'target',
           'request_ts', 'hora', 'mes', 'dia_semana', 'trimestre']
num_cols = df_master.select_dtypes(include=[np.number]).columns.tolist()
features = [c for c in num_cols if c not in excluir and not c.startswith('lista_')]
print(f"Total de features: {len(features)}")

X = df_master[features].fillna(0).values
y = df_master['target'].values

# SPLIT TEMPORAL — df_master ja esta ordenado por time_window
corte = int(len(df_master) * 0.80)
X_train_raw, X_test_raw = X[:corte], X[corte:]
y_train, y_test = y[:corte], y[corte:]

print(f"Treino (80% inicial): {len(X_train_raw):,} | Teste (20% final): {len(X_test_raw):,}")

# Normalizacao (fit apenas no treino, para nao vazar estatisticas do teste)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

# One-hot do target
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=3)
y_test_cat  = tf.keras.utils.to_categorical(y_test,  num_classes=3)

print("\nDistribuicao das classes no TESTE:")
print(pd.Series(y_test).map(target_map).value_counts())

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Activation, Input
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

print("--- Construindo a rede neural otimizada ---")

def bloco(units, drop):
    # Ordem canonica: Dense (sem bias) -> BatchNorm -> ReLU -> Dropout
    return [Dense(units, use_bias=False), BatchNormalization(), Activation('relu'), Dropout(drop)]

camadas  = [Input(shape=(X_train_scaled.shape[1],))]
camadas += bloco(256, 0.4)
camadas += bloco(128, 0.3)
camadas += bloco(64,  0.2)
camadas += [Dense(3, activation='softmax')]
model_dl = Sequential(camadas)

# Pesos de classe (leve — dados ja balanceados 1:1:1)
pesos = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight = {i: w for i, w in enumerate(pesos)}

# AdamW com weight decay; fallback para Adam em versoes antigas
try:
    opt = tf.keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=1e-4)
except Exception:
    opt = tf.keras.optimizers.Adam(learning_rate=1e-3)

loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05)
model_dl.compile(optimizer=opt, loss=loss, metrics=['accuracy'])

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-5, verbose=1),
]

print("Treinando (val = ultimos 15% do treino, temporalmente coerente)...")
hist = model_dl.fit(
    X_train_scaled, y_train_cat,
    epochs=150, batch_size=512, validation_split=0.15,
    class_weight=class_weight, callbacks=callbacks, verbose=1,
)
print("\n✅ Treino concluido.")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns

nomes = ['Nao Evento (0)', 'Clima (1)', 'Atraso Voo (2)']
y_prob = model_dl.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

print("="*55)
print("RESULTADOS — DEEP LEARNING OTIMIZADO")
print("="*55)
print(classification_report(y_test, y_pred, target_names=nomes, digits=4, zero_division=0))
print(f"Macro F1: {f1_score(y_test, y_pred, average='macro'):.4f}")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=nomes, yticklabels=nomes)
plt.title('Matriz de Confusao — DL Otimizado')
plt.ylabel('Classe Real'); plt.xlabel('Classe Prevista')
plt.tight_layout(); plt.show()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(hist.history['loss'],     label='treino')
ax[0].plot(hist.history['val_loss'], label='validacao')
ax[0].set_title('Loss'); ax[0].set_xlabel('epoca'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(hist.history['accuracy'],     label='treino')
ax[1].plot(hist.history['val_accuracy'], label='validacao')
ax[1].set_title('Acuracia'); ax[1].set_xlabel('epoca'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 🔍 Explicabilidade (XAI) com SHAP

Quais features mais empurram a decisão do modelo para cada classe de evento.

In [ ]:
!pip install -q shap
import shap
import numpy as np

print("Calculando valores SHAP (DeepExplainer)...")
background = X_train_scaled[np.random.choice(X_train_scaled.shape[0], 200, replace=False)]
explainer  = shap.DeepExplainer(model_dl, background)

n = 400
shap_values = explainer.shap_values(X_test_scaled[:n])

# Compatibilidade entre versoes do SHAP
if isinstance(shap_values, list):
    sv = {1: shap_values[1], 2: shap_values[2]}
elif hasattr(shap_values, 'shape') and len(shap_values.shape) == 3:
    sv = {1: shap_values[:, :, 1], 2: shap_values[:, :, 2]}
else:
    sv = {1: shap_values, 2: shap_values}

X_disp = df_master[features].iloc[corte:corte + n]

print("\n🔍 FATORES QUE MAIS IMPACTAM EVENTO CLIMATICO (Classe 1):")
shap.summary_plot(sv[1], X_disp, feature_names=features, plot_size=(10, 6))

print("\n🔍 FATORES QUE MAIS IMPACTAM ATRASO DE VOO (Classe 2):")
shap.summary_plot(sv[2], X_disp, feature_names=features, plot_size=(10, 6))